In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

set_random_seed = 171

# Load datasets
train_df = pd.read_csv("data/train_dataset.csv")
val_df = pd.read_csv("data/val_dataset.csv")

# Separate features and target
X_train = train_df.drop(columns=["Target_binary"])
y_train = train_df["Target_binary"]

X_val = val_df.drop(columns=["Target_binary"])
y_val = val_df["Target_binary"]

print("Datasets loaded successfully.")

print("\nTraining shape:", X_train.shape)
print("Test shape:", X_val.shape)

Datasets loaded successfully.

Training shape: (2654, 20)
Test shape: (663, 20)


In [2]:
# Random Forest model
best_model = RandomForestClassifier(
    n_estimators=300,
    random_state=set_random_seed,
    class_weight="balanced"
)

# Cross-validation ONLY on training data
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=set_random_seed)

scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

cv_scores = cross_validate(
    best_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

In [3]:
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_val)

print("\nVal Set Performance (15% Super Unseen Data)")
print("Precision:", precision_score(y_val, y_pred))
print("Recall:", recall_score(y_val, y_pred))
print("F1:", f1_score(y_val, y_pred))


Val Set Performance (15% Super Unseen Data)
Precision: 0.8597560975609756
Recall: 0.6619718309859155
F1: 0.7480106100795756


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    classification_report,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
)

# Probabilities / scores (needed for ROC / PR / calibration metrics)
# RandomForest supports predict_proba
y_proba = best_model.predict_proba(X_val)[:, 1]

# Confusion Matrix (default threshold = 0.5)
cm = confusion_matrix(y_val, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Dropout", "Dropout"])
disp.plot(values_format="d")
plt.title("Confusion Matrix (threshold=0.5)")
plt.show()

# Classification report (precision/recall/f1 per class)
print("\n=== Classification Report (Test Set) ===")
print(classification_report(y_val, y_pred, target_names=["Not Dropout", "Dropout"]))

# ROC Curve + AUC
roc_auc = roc_auc_score(y_val, y_proba)
RocCurveDisplay.from_predictions(y_val, y_proba)
plt.title(f"ROC Curve (AUC = {roc_auc:.4f})")
plt.show()

# Precision-Recall Curve + Average Precision (often more informative with imbalance)
avg_prec = average_precision_score(y_val, y_proba)
PrecisionRecallDisplay.from_predictions(y_val, y_proba)
plt.title(f"Precision-Recall Curve (AP = {avg_prec:.4f})")
plt.show()

# Additional summary metrics
print("\n=== Additional Metrics (Test Set) ===")
print(f"ROC AUC:              {roc_auc:.4f}")
print(f"Average Precision:    {avg_prec:.4f}")
print(f"Balanced Accuracy:    {balanced_accuracy_score(y_val, y_pred):.4f}")

# Threshold sweep: see how precision/recall/f1 change with cutoff
thresholds = np.linspace(0.05, 0.95, 19)

def prf_at_threshold(t):
    pred_t = (y_proba >= t).astype(int)
    # Manual computation to avoid extra imports; safe for binary labels 0/1
    cm_t = confusion_matrix(y_val, pred_t, labels=[0, 1])
    tn, fp, fn, tp = cm_t.ravel()
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return precision, recall, f1

precisions, recalls, f1s = [], [], []
for t in thresholds:
    p, r, f = prf_at_threshold(t)
    precisions.append(p); recalls.append(r); f1s.append(f)

best_idx = int(np.argmax(f1s))
best_t = float(thresholds[best_idx])

plt.plot(thresholds, precisions, label="Precision")
plt.plot(thresholds, recalls, label="Recall")
plt.plot(thresholds, f1s, label="F1")
plt.axvline(best_t, linestyle="--", label=f"Best F1 threshold={best_t:.2f}")
plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.title("Threshold Sweep on Test Set")
plt.legend()
plt.show()

print(f"\nBest F1 on test set occurs at threshold ≈ {best_t:.2f} with F1 = {f1s[best_idx]:.4f}")

# Feature importance (helpful for interpretability; note: RF importances can be biased toward high-cardinality) 
importances = best_model.feature_importances_
order = np.argsort(importances)[::-1]

top_k = min(20, len(importances))
plt.figure(figsize=(8, 6))
plt.barh(range(top_k), importances[order][:top_k][::-1])
plt.yticks(range(top_k), [X_val.columns[i] for i in order[:top_k]][::-1])
plt.xlabel("Importance")
plt.title(f"Top {top_k} Feature Importances (Random Forest)")
plt.tight_layout()
plt.show()